In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/Tommachilez/fact-checking-data-framework-vn.git
%cd /content/fact-checking-data-framework-vn/

Cloning into 'fact-checking-data-framework-vn'...
remote: Enumerating objects: 635, done.
remote: Counting objects: 100% (291/291), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 635 (delta 200), reused 153 (delta 83), pack-reused 344 (from 1)
Receiving objects: 100% (635/635), 5.39 MiB | 16.22 MiB/s, done.
Resolving deltas: 100% (424/424), done.
/content/fact-checking-data-framework-vn


In [6]:
from google.colab import userdata

GEMINI_KEY = userdata.get('GEMINI_1.5')

dataset_path = "/content/drive/MyDrive/KLTN_Project/Datasets"
doc_mapping = f"{dataset_path}/unique_doc_mapping.csv"
query_mapping = f"{dataset_path}/test_query_mapping.csv"
output_dir = f"{dataset_path}/gemini_labeling_output_hybrid"
vifc_test = f"{dataset_path}/vifc_test_set_with_labels.csv"

dp_run_file = f"{dataset_path}/deeperimpact_vifc/run_file.txt"
bm25_run_file = f"{dataset_path}/bm25_run_file.txt"
test_qrels = f"{dataset_path}/test_qrels.tsv"
run_file = f"{dataset_path}/hybrid_run_file.txt"

In [4]:
!python -m src.scripts.score_hybrid \
      --bm25_run {bm25_run_file} \
      --deep_impact_run {dp_run_file} \
      --output_file {run_file} \
      --alpha 0.8 \
      --normalize

Reading /content/drive/MyDrive/KLTN_Project/Datasets/bm25_run_file.txt...
Reading /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact_vifc/run_file.txt...
Normalizing scores (Min-Max)...
Calculating Hybrid Scores with alpha=0.8...
Processing queries: 100% 7685/7685 [00:06<00:00, 1124.97it/s]
Writing results to /content/drive/MyDrive/KLTN_Project/Datasets/hybrid_run_file.txt...
Hybrid run file created successfully.


In [7]:
!python -m src.scripts.rag_inference \
    --run_file {run_file} \
    --doc_mapping {doc_mapping} \
    --query_mapping {query_mapping} \
    --label_file {vifc_test} \
    --api_key {GEMINI_KEY} \
    --output_dir {output_dir} \
    --top_k 3 \
    # --qid '2216'

>>> Initializing Reader LLM with model: gemini-2.5-flash...
Loading document text from /content/drive/MyDrive/KLTN_Project/Datasets/unique_doc_mapping.csv...
Loading Docs: 3030it [00:00, 13535.71it/s]
Loading queries from /content/drive/MyDrive/KLTN_Project/Datasets/test_query_mapping.csv...
Loading top-3 results from /content/drive/MyDrive/KLTN_Project/Datasets/hybrid_run_file.txt...
Loading labels from /content/drive/MyDrive/KLTN_Project/Datasets/vifc_test_set_with_labels.csv...
Mapped 7685 ground truth labels to Query IDs.
Processing Queries:   2% 125/7685 [11:30<13:20:00,  6.35s/it]Error generating answer for ID 27143: Expecting ',' delimiter: line 5 column 777 (char 1126)
Processing Queries:   2% 144/7685 [13:35<11:13:00,  5.35s/it]Error generating answer for ID 27162: Expecting ',' delimiter: line 5 column 544 (char 791)
Processing Queries:   3% 264/7685 [24:54<8:33:06,  4.15s/it]Error generating answer for ID 27282: Expecting ',' delimiter: line 5 column 403 (char 675)
Processin